# Bloque A en Kaggle: **YOLO26** (A0 y A3), Holes y Spots

Sugerencia del profesor: cambiar YOLO11 por YOLO26. Este lanzador ejecuta el notebook
`notebooks/02_baseline_supervisado_cisutac.ipynb` con `MODELO = "yolo26n.pt"` y **sólo las dos variantes que importan**:

- **A0:** foto completa a 640 (baseline).
- **A3:** entrenar e inferir con tiles de 640 y 20 % de solapamiento. Usa la misma proporción de tiles sanos que la mejor
  variante de YOLO11 en valid: `neg1` en Holes y `neg0` en Spots. Así la comparación YOLO11 vs. YOLO26 es directa.

Se apagan A1 (1280) y las ablaciones de A3. A2 (el modelo A0 sobre tiles) queda porque sólo es inferencia.
Corre primero Holes y después Spots en la misma sesión. **Sólo valid:** el test sigue cerrado
(`EVALUAR_TEST = False`) hasta decidir con valid si YOLO26 reemplaza a YOLO11.

**Settings (panel derecho):** *Accelerator* = **GPU T4 x2** (o P100) · *Internet* = **On**.

1. (Opcional) Verificar con `MODO_PRUEBA = True` y **Run All** (unos minutos).
2. Corrida real: `MODO_PRUEBA = False` y **Save Version → Save & Run All (Commit)**. Se puede cerrar la compu.
3. Al terminar, en *Output*: `results_bloque_a_yolo26.zip` (descomprimir en la raíz del repo) y un notebook ejecutado
   por dataset, `02_baseline_supervisado_cisutac_ejecutado_<dataset>_yolo26.ipynb`.

Si una corrida se corta, agregar su Output con **Add Input** y volver a correr: se reutiliza lo ya entrenado.
Las corridas se guardan con el sufijo `_yolo26n`, así que no se mezclan con las de YOLO11.


In [ ]:
# ===== Parámetros: lo único que hay que tocar =====
DATASETS = ["holes", "spots"]   # se corren uno detrás del otro
MODO_PRUEBA = False             # True: corrida corta para verificar que todo anda. False: corrida real
MODELO = "yolo26n.pt"           # YOLO26 nano preentrenado en COCO
RAMA = "main"                   # rama del repo a usar

# Proporción de tiles sanos de A3: la de la mejor variante de YOLO11 en valid de cada dataset
NEG_POR_POS = {"holes": 1.0, "spots": 0.0}


In [ ]:
NOTEBOOK = "02_baseline_supervisado_cisutac.ipynb"
ZIP_RESULTADOS = "results_bloque_a_yolo26"
PAQUETES = "-U ultralytics sahi pycocotools papermill 'huggingface_hub[cli]'"   # -U: YOLO26 necesita ultralytics reciente

import os, subprocess, sys, shutil, glob, json
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "tp_final_vision"
DATA = Path("/tmp/rewear_data")   # fuera de /kaggle/working: los datos no ocupan lugar en el Output
os.environ["REWEAR_DATA_DIR"] = str(DATA)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not REPO.exists():
    sh(f"git clone --depth 1 -b {RAMA} https://github.com/victoriapersico/tp_final_vision.git {REPO}")
else:
    sh(f"git -C {REPO} pull")
sh(f"git -C {REPO} log --oneline -1")
sh(f"{sys.executable} -m pip install -q {PAQUETES}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")


In [ ]:
# Si se agregó como Input el Output de una versión anterior de este notebook, se recuperan sus resultados
# para no repetir lo que ya se calculó (los notebooks reutilizan todo lo que encuentran en results/).
for prev in glob.glob("/kaggle/input/**/tp_final_vision/results", recursive=True):
    print("Recuperando resultados previos de", prev)
    shutil.copytree(prev, REPO / "results", dirs_exist_ok=True)
for z in glob.glob(f"/kaggle/input/**/{ZIP_RESULTADOS}.zip", recursive=True):
    print("Recuperando resultados previos de", z)
    shutil.unpack_archive(z, REPO)


In [ ]:
# CISUTAC Holes y Spots (~230 MB cada uno) y split agrupado por día (reports/cisutac/cis5_split_propuesto.csv).
sh(f"cd {REPO} && {sys.executable} scripts/download_datasets.py all")
sh(f"cd {REPO} && {sys.executable} scripts/prepare_cisutac_yolo.py")


In [ ]:
# Copia del notebook 02 con los parámetros fijados y ejecución de punta a punta, una vez por dataset.
# Se reemplazan las líneas que empiezan con "<NOMBRE> =" en cualquier celda de código; las listas vacías
# apagan A1 y las ablaciones de A3 (sus celdas corren igual, pero sin entrenar nada).
import papermill as pm

def preparar(dataset):
    parametros = {"DATASET": dataset, "MODO_PRUEBA": MODO_PRUEBA, "EVALUAR_TEST": False, "MODELO": MODELO,
                  "NEG_POR_POS": NEG_POR_POS[dataset],
                  "RESOLUCIONES_A1": [], "ABLACION_NEGATIVOS": [], "ABLACION_TILES": []}
    nb = json.loads((REPO / "notebooks" / NOTEBOOK).read_text(encoding="utf-8"))
    encontrados = {nombre: 0 for nombre in parametros}
    for celda in nb["cells"]:
        if celda["cell_type"] != "code":
            continue
        lineas = "".join(celda["source"]).splitlines(keepends=True)
        for i, linea in enumerate(lineas):
            for nombre, valor in parametros.items():
                if linea.startswith(f"{nombre} ="):
                    lineas[i] = f"{nombre} = {valor!r}  # fijado por el lanzador de Kaggle" + chr(10)
                    encontrados[nombre] += 1
        celda["source"] = "".join(lineas)
    faltan = [n for n, k in encontrados.items() if k != 1]
    assert not faltan, f"El notebook 02 cambió: no encuentro una única línea para {faltan}"
    entrada = REPO / "notebooks" / f"_kaggle_{dataset}_{NOTEBOOK}"
    entrada.write_text(json.dumps(nb, ensure_ascii=False), encoding="utf-8")
    return entrada

try:
    for dataset in DATASETS:
        entrada = preparar(dataset)
        salida = WORK / f"{Path(NOTEBOOK).stem}_ejecutado_{dataset}_{Path(MODELO).stem.rstrip('n')}{'_prueba' if MODO_PRUEBA else ''}.ipynb"
        print(f"===== {dataset}: {entrada.name} -> {salida.name}", flush=True)
        pm.execute_notebook(str(entrada), str(salida), cwd=str(REPO / "notebooks"), log_output=True,
                            kernel_name="python3")
finally:
    # Aunque falle a mitad de camino, se empaqueta lo que se haya llegado a calcular.
    shutil.make_archive(str(WORK / ZIP_RESULTADOS), "zip", REPO, "results")
    print("Resultados en", WORK / f"{ZIP_RESULTADOS}.zip")
